# Qmod Tutorial — Part II

Welcome back to the Qmod tutorial! **Part I** introduced the language foundations —
quantum variables and types, output parameters, classical and quantum control flow, and quantum
arithmetic. **Part II** (this notebook) builds on those foundations to cover the statements and
operators used to express real quantum algorithms.

Six sections, each pairing a concept with a hands-on exercise:

7. Phase statements
8. Local variables, permutations, and auto-uncomputation
9. Within-apply
10. Hamiltonians, Pauli operators, and exponentiation
11. Higher-order functions
12. Execution parameters and hybrid execution

Each section has a short concept explanation followed by an exercise. Complete the code where
marked with `# TODO`, then synthesize and run it. **All solutions are
collected at the end of the notebook** — try each exercise before peeking.

> **Note:** If you are working in your own SDK environment, make sure Classiq is installed
> (`pip install -U classiq`) and that you have authenticated once by running `authenticate()`
> in a Python session. See the
> [registration and installation guide](https://docs.classiq.io/latest/getting-started/registration_installations/).

In [ ]:
# Run once per environment to authenticate:
# import classiq
# classiq.authenticate()

from classiq import *
from classiq.qmod.symbolic import pi

## 7. Phase statements

The [`phase`](https://docs.classiq.io/qmod-reference/language-reference/statements/phase) statement applies phases to computational-basis states, where the Z-axis rotation can be a function of quantum numeric variables. In ket notation the operator maps $|x_1, x_2, \dots\rangle$ to $e^{\,i\,f(x_1, x_2, \dots)}\,|x_1, x_2, \dots\rangle$, where $f$ is an expression over the variables $x_1, x_2, \dots$

For example, the following rotates each state by $\frac{\pi}{4}x^2$:
```python
phase(pi / 4 * x**2)
```
If $x$ is a `QNum` in the range $[0, 1, 2, 3]$ it rotates $|1\rangle$ by $\pi/4$, $|2\rangle$ by $4\pi/4$ (i.e., half turn), and $|3\rangle$ by $9\pi/4$ (coming back to $\pi/4$).

> **Note**: An optional second argument multiplies the expression by a classical coefficient (Part I, §3) — typically an execution parameter, as in a QAOA cost layer.

When the phase expression contains no quantum variables, the same phase is applied across all states. Still, this can be restricted to certain states by an enclosing `control` statement.

For example, if $x$ is a `QNum`, the following rotates all states other than $|0\rangle$ by $\pi/4$:
```python
control(x!=0, lambda: phase(pi / 4))
```

Phase statements are the workhorse behind **phase oracles** (as in Grover's algorithm) and **cost functions** (as in QAOA).

> **Note**: Phases are invisible to ordinary sampling — they don't change measurement probabilities. To "see" them, use the top-level `calculate_state_vector(qprog)`, which returns the amplitude (including phase) of each state.

**Exercise A:** Put two 2-qubit numbers `x` and `y` into uniform superposition, then encode their product `x * y` into the phase, scaled by a coefficient of $2\pi / 2^N = \pi/8$ (with $N = 4$ the total number of qubits). This divides the full turn into $2^N = 16$ equal steps — one per unit of the product — so distinct product values land on distinct phases. Use `calculate_state_vector` to inspect the resulting phases.

In [ ]:
@qfunc
def main(x: Output[QNum[2]], y: Output[QNum[2]]):
    allocate(x)
    allocate(y)
    hadamard_transform(x)
    hadamard_transform(y)

    # TODO: encode the product x * y into the phase, using the coefficient given above


qprog = synthesize(main)
show(qprog)

result = calculate_state_vector(qprog)
display(result)

> **Expected result:** every basis state has probability 1/16, but the `phase` column shows a rotation of $(x \cdot y)\,\pi/8$ for each state — e.g. `x=2, y=1` gives $0.25\pi$, and `x=3, y=3` gives $9\pi/8$ (shown as $-0.88\pi$, i.e. modulo $2\pi$).

**Exercise B:** A **phase oracle** flips the phase (a rotation by $\pi$) of the basis states that satisfy some condition — the marking step at the heart of Grover's algorithm. Using two 2-qubit numbers `a` and `b` in uniform superposition, complete `mark_solutions` to flip the phase of every state satisfying $3a + b = 9$ (similar to the exercise in Part I, §6). Then inspect the statevector to confirm which states were marked.

In [ ]:
@qfunc
def mark_solutions(a: QNum, b: QNum):
    # TODO: flip the phase (by pi) of the states satisfying 3*a + b == 9,
    #       by conditioning a phase statement on that equation
    pass


@qfunc
def main(a: Output[QNum[2]], b: Output[QNum[2]]):
    allocate(a)
    allocate(b)
    hadamard_transform(a)
    hadamard_transform(b)
    mark_solutions(a, b)


qprog = synthesize(main)
show(qprog)

result = calculate_state_vector(qprog)
display(result)

> **Expected result:** all 16 basis states keep probability 1/16, but exactly two of them — the assignments satisfying $3a+b=9$: `(a, b) = (3, 0)` and `(2, 3)` — show a phase of `π`; every other state shows `0`.

## 8. Local variables, permutations, and auto-uncomputation

Inside a quantum function you can declare a **local** quantum variable and use it to hold intermediate results. This is very handy in breaking computations into smaller steps. When a local goes out of scope, Qmod **uncomputes** it automatically — reversing the operations that set it, so its qubits return to $|0\rangle$ and can be reused.

For example, the following declares a numeric local variable `v` and assigns it the computational-basis value 3.
```python
v = QNum()
v |= 3
```

Uncomputation can work as long as the local variable was modified by **permutations** — operations that map basis states to basis states without creating superposition, such as gate-level functions `X` and `CX`, as well as higher level arithmetic. If a local can't be uncomputed automatically the compiler will issue an error. You can discard the variable explicitly with `drop` or manually uncompute it and call `free`. See [uncomputation](https://docs.classiq.io/qmod-reference/language-reference/uncomputation) for the full rules.

**Exercise:** Put a 2-qubit number `x` into uniform superposition. Use a *local* variable `tmp` to store $2x+1$, and depending on `tmp > 3`, flip `res`. `tmp` is a scratch variable — it is uncomputed automatically at the end of the function, so it never appears in the output.

In [ ]:
@qfunc
def main(x: Output[QNum], res: Output[QBit]):
    allocate(2, x)
    hadamard_transform(x)
    allocate(res)

    # TODO: store 2 * x + 1 in a local QNum tmp using assignment ('|=')
    #       then flip res for the states where tmp > 3
    # (tmp is a scratch variable — it is uncomputed automatically)


qprog = synthesize(main)
show(qprog)

result = sample(qprog)
display(result)

> **Expected result:** `x` is uniform over 0–3, and `res` is `1` exactly when `tmp = 2x + 1 > 3` — that is, when `x > 1` (`x` is 2 or 3). The scratch variable `tmp` was uncomputed automatically, so the output contains only `x` and `res`.

## 9. Within-apply

Many quantum routines are based on the conjugation pattern: transform the state with some operation $U$, perform an operation $V$ in that transformed basis, then transform back, with the net effect – $U^\dagger V U$. The [within-apply](https://docs.classiq.io/qmod-reference/language-reference/statements/within-apply) statement captures this directly. When this pattern is subject to quantum control, only the `apply` block (the $V$ operation) actually needs to be controlled.

For example, the following applies `Z` to a qubit `q` in the Hadamard basis (which is equivalent to applying `X` to `q`):
```python
within_apply(lambda: H(q), lambda: Z(q))
```

> **Note**: Initializing a variable in the `within` block implies that its inverse uncomputes and frees it. Hence, such a variable is subject to the same rules as local variables we discussed in §8.

**Exercise A:** A classic example of conjugation is performing addition in the Fourier basis by modifying relative phases. Complete the `within_apply` so it computes `y += x`. Variable `x` starts in uniform superposition and `y` at the fixed value 2, so each measured `y` should come out as `x + 2`.

In [ ]:
@qfunc
def main(x: Output[QNum[3]], y: Output[QNum[4]]):
    allocate(x)
    hadamard_transform(x)
    y |= 2  # fixed starting value

    # TODO: compute y += x in the Fourier basis. Put qft(y) in the within block; the apply block is a
    #       phase rotating by x * y, scaled so one full wrap of y is a single turn (i.e. 2 * pi / 2**y.size)


qprog = synthesize(main)
show(qprog)

result = sample(qprog)
display(result)

> **Expected result:** `x` is uniform over 0–7, and each row's `y` equals `x + 2` (so `y` ranges 2–9) — the in-place addition carried out entirely through phase rotations in the Fourier basis.

**Exercise B:** Now perform the *same* addition, but only when a control qubit `ctrl` is $|1\rangle$ — put the whole `within_apply` under a `control` on `ctrl`. Then run `show(qprog)` and inspect the quantum program: notice that the `qft` and its inverse are **not** controlled — only the `apply` (phase) block is.

In [ ]:
@qfunc
def main(ctrl: Output[QBit], x: Output[QNum[3]], y: Output[QNum[4]]):
    allocate(ctrl)
    hadamard_transform(ctrl)  # superpose ctrl so both cases appear
    allocate(x)
    hadamard_transform(x)
    y |= 2  # fixed starting value

    # TODO: perform the same Fourier-basis y += x, but only when ctrl is |1>
    #       (put the whole within_apply under a control on ctrl)


qprog = synthesize(main)
show(qprog)

result = sample(qprog)
display(result)

> **Expected result:** when `ctrl` is `0`, `y` stays `2`; when `ctrl` is `1`, `y` equals `x + 2`. In the visualization, the `qft` and inverse-`qft` surround the phase block but sit *outside* the control.

## 10. Hamiltonians, Pauli operators, and exponentiation

The four **Pauli matrices** `I`, `X`, `Y`, `Z` are the building blocks of qubit operators. A **Hamiltonian** — a system's energy operator — can be represented as a weighted sum of Pauli matrix products. In Qmod this is called **Pauli operator**, and you build it from the `Pauli` enum, where `Pauli.X(0)` is `X` on qubit 0; multiply to form a product (unmentioned qubits are implicitly `I`), and add to sum terms.

Here is an example Hamiltonian:

```python
1.0 * Pauli.X(0) + 0.2 * Pauli.Y(0) * Pauli.Z(1) + 0.5 * Pauli.Z(1)
```

A Hamiltonian drives time evolution through $U(t) = e^{-iHt}$. Exact gate decomposition of this operator can be exponentially large, so the **Suzuki-Trotter** decomposition approximates it as `r` repeated short steps over the individual terms. The built-in function [`suzuki_trotter`](https://docs.classiq.io/qmod-reference/library-reference/core-library-functions/hamiltonian_evolution/suzuki_trotter/suzuki_trotter) applies it, given the Hamiltonian, the evolution time, the order, and the repetitions `r`.

A Pauli operator can also serve as an **observable** — a quantity whose expectation value $\langle\psi|O|\psi\rangle$ we measure. Where `sample` returns the distribution over basis states, the SDK function `observe` returns the expectation value of a given observable.

**Exercise A:** Use `suzuki_trotter` to simulate evolution under

$$H = 0.5\,X_0 X_1 Z_2 X_3 \;+\; 0.25\,Z_1 Y_3$$

on 4 qubits, with evolution time $t = 3$, 2nd order, and $r = 2$ repetitions.

> **Tip**: Avoid declaring a Python variable with the name `H` in the global scope, because it would eclipse the function `H` (the Hadamard function)

In [ ]:
@qfunc
def main(qarr: Output[QArray[QBit]]):
    allocate(4, qarr)

    # TODO: build H above as a Pauli expression and apply suzuki_trotter
    #       (evolution time 3, order 2, 2 repetitions, acting on qarr)


qprog = synthesize(main)
show(qprog)

**Exercise B:** Prepare two qubits in $|+\rangle|+\rangle$ (with `hadamard_transform`), then use `observe` to compute the expectation value of the observable $X_0 + X_1$.

In [ ]:
@qfunc
def main(qarr: Output[QArray[QBit, 2]]):
    allocate(qarr)
    hadamard_transform(qarr)  # each qubit in |+>


qprog = synthesize(main)
show(qprog)

# TODO: build the observable X0 + X1 as a Pauli expression,
#       then compute its expectation value with observe(qprog, ...)

> **Expected result:** `observe` returns `2.0` — each qubit is in $|+\rangle$, the $+1$ eigenstate of `X`, so $\langle X_0\rangle = \langle X_1\rangle = 1$.

## 11. Higher-order functions

A **higher-order function** is a quantum function that takes another quantum function as an argument. A function-type parameter is declared with type `QCallable` (the Qmod analog of Python's `Callable`). When calling a higher-order function, you pass a function or a lambda expression, whose signature must match the one specified by the `QCallable` type.

For example, the following declares `foo` with a function-type parameter `op`; calling `foo` requires passing a function that takes a classical real and a qubit:
```python
@qfunc
def foo(op: QCallable[CReal, QBit], q: QBit):
    ...
```

Higher-order functions are very useful for capturing recurring patterns in quantum algorithms. Examples are the Quantum Phase Estimation (QPE) and the Grover double-reflection operator.

> **Note:** Passing a lambda to a higher-order function works just like passing one to a built-in statement — e.g. `repeat(qarr.len, lambda i: H(qarr[i]))`, where `i` is the lambda's CInt parameter (Part I, §4).

See more under [operators](https://docs.classiq.io/qmod-reference/language-reference/operators).

**Exercise A:** Define `my_apply_to_all` — a higher-order function that applies a single-qubit operation `op` to every qubit of `qarr`. Then use it to apply `H` to all three qubits, producing a uniform superposition.

In [ ]:
@qfunc
def my_apply_to_all(op: QCallable[QBit], qarr: QArray[QBit]):
    # TODO: apply op to each qubit of qarr
    pass


@qfunc
def main(qarr: Output[QArray[QBit, 3]]):
    allocate(qarr)
    my_apply_to_all(lambda t: H(t), qarr)


qprog = synthesize(main)
show(qprog)

result = sample(qprog)
display(result)

> **Expected result:** all eight bit strings appear with roughly equal ~1/8 probability — `H` was applied to every qubit.

**Exercise B:** [`qpe`](https://docs.classiq.io/qmod-reference/library-reference/open-library-functions/qpe/qpe) (quantum phase estimation) is a built-in higher-order function: given a unitary `U` as its operand, it estimates the phase $\theta$ of an eigenstate, where $U|\psi\rangle = e^{2\pi i\theta}|\psi\rangle$, and writes it into a `QNum`. Here `|11⟩` is an eigenstate of `CRZ(pi, state[0], state[1])` with $\theta = 0.25$. Complete the `qpe` call to estimate it.

In [ ]:
@qfunc
def main(theta: Output[QNum[4, UNSIGNED, 4]]):
    state = QArray()
    allocate(2, state)
    X(state[0])
    X(state[1])  # prepare |11>, an eigenstate of the unitary below
    allocate(theta)

    # TODO: call qpe, passing CRZ(pi, state[0], state[1]) as its unitary operand
    #       and theta as the output phase variable


qprog = synthesize(main)
show(qprog)

result = sample(qprog)
display(result)

> **Expected result:** `theta` comes out `0.25` with near-certainty — the phase of `CRZ(pi)` on its eigenstate `|11⟩`.

## 12. Execution parameters and hybrid execution

Hybrid algorithms (VQE, QAOA, ...) split work between a quantum program and a classical loop that tunes the program's parameters. A classical parameter of `main` — declared with type `CReal`, `CArray[CReal]`, and so on — is an **execution parameter**: its value is supplied at run time, so one synthesized program runs at many parameter values without re-synthesizing.

You pass the values through the `parameters=` argument of the top-level `sample` or `observe` — a dict mapping each parameter name to its value:

```python
sample(qprog, parameters={"params": [0.2, 1.0]})
```

For all execution options, see [execution](https://docs.classiq.io/user-guide/execution/index).

> **Note:** Execution parameters are ordinary numbers, so use Python floats (e.g. `math.pi`), not the symbolic `pi` used inside function bodies.

**Exercise A:** The parametric program below rotates two qubits by `params[0]` and `params[1]`, then entangles them. Synthesize it once, then use `sample` to run it at `params = [0.2, 1.0]`.

In [ ]:
@qfunc
def main(params: CArray[CReal, 2], qarr: Output[QArray[QBit]]):
    allocate(2, qarr)
    RY(params[0], qarr[0])
    RY(params[1], qarr[1])
    CX(qarr[0], qarr[1])


qprog = synthesize(main)
show(qprog)

# TODO: sample qprog at params = [0.2, 1.0], supplied via the parameters= argument

> **Expected result:** the distribution is dominated by `00`, with `10` the main other outcome — set by the two `RY` angles and the entangling `CX`. Re-running with different `params` changes the distribution, reusing the *same* synthesized `qprog`.

**Exercise B:** Using the same `qprog` from *Exercise A*, compute the expectation value $\langle H \rangle$ of $H = X_0 + 0.5\, Z_0 Z_1$ at parameter values $[\pi/4,\, \pi/3]$ with `observe`. Remember to pass the angles as numbers.

In [ ]:
import math

hamiltonian = Pauli.X(0) + 0.5 * Pauli.Z(0) * Pauli.Z(1)

# TODO: compute <H> at params = [math.pi/4, math.pi/3] with observe(qprog, hamiltonian, parameters=...)

> **Expected result:** `observe` returns a single number, $\langle H\rangle \approx 0.86$ — the expectation value on the state prepared at those parameters.

## Solutions

Try each exercise before checking the solution below.

### Solution 7 — Phase statements

**Exercise A — encoding a product into the phase**

In [ ]:
@qfunc
def main(x: Output[QNum[2]], y: Output[QNum[2]]):
    allocate(x)
    allocate(y)
    hadamard_transform(x)
    hadamard_transform(y)
    phase(2 * pi / 2 ** (x.size + y.size) * x * y)


qprog = synthesize(main)
result = calculate_state_vector(qprog)
display(result)

**Exercise B — a Grover-style phase oracle**

In [ ]:
@qfunc
def mark_solutions(a: QNum, b: QNum):
    control(3 * a + b == 9, lambda: phase(pi))


@qfunc
def main(a: Output[QNum[2]], b: Output[QNum[2]]):
    allocate(a)
    allocate(b)
    hadamard_transform(a)
    hadamard_transform(b)
    mark_solutions(a, b)


qprog = synthesize(main)
result = calculate_state_vector(qprog)
display(result)

### Solution 8 — Local variables, permutations, and auto-uncomputation

In [ ]:
@qfunc
def main(x: Output[QNum], res: Output[QBit]):
    allocate(2, x)
    hadamard_transform(x)
    allocate(res)

    tmp = QNum()
    tmp |= 2 * x + 1
    control(tmp > 3, lambda: X(res))


qprog = synthesize(main)
result = sample(qprog)
display(result)

### Solution 9 — Within-apply

**Exercise A — within-apply addition**

In [ ]:
@qfunc
def main(x: Output[QNum[3]], y: Output[QNum[4]]):
    allocate(x)
    hadamard_transform(x)
    y |= 2  # fixed starting value
    within_apply(
        lambda: qft(y),
        lambda: phase(2 * pi / 2**y.size * x * y),
    )


qprog = synthesize(main)
result = sample(qprog)
display(result)

**Exercise B — the same addition under control**

In [ ]:
@qfunc
def main(ctrl: Output[QBit], x: Output[QNum[3]], y: Output[QNum[4]]):
    allocate(ctrl)
    hadamard_transform(ctrl)
    allocate(x)
    hadamard_transform(x)
    y |= 2  # fixed starting value
    control(
        ctrl,
        lambda: within_apply(
            lambda: qft(y),
            lambda: phase(2 * pi / 2**y.size * x * y),
        ),
    )


qprog = synthesize(main)
result = sample(qprog)
display(result)

### Solution 10 — Hamiltonians, Pauli operators, and exponentiation

**Exercise A — Suzuki-Trotter evolution**

In [ ]:
@qfunc
def main(qarr: Output[QArray[QBit]]):
    allocate(4, qarr)
    suzuki_trotter(
        0.5 * Pauli.X(0) * Pauli.X(1) * Pauli.Z(2) * Pauli.X(3)
        + 0.25 * Pauli.Z(1) * Pauli.Y(3),
        evolution_coefficient=3,
        order=2,
        repetitions=2,
        qbv=qarr,
    )


qprog = synthesize(main)
result = sample(qprog)
display(result)

**Exercise B — measuring an observable with `observe`**

In [ ]:
@qfunc
def main(qarr: Output[QArray[QBit, 2]]):
    allocate(qarr)
    hadamard_transform(qarr)


qprog = synthesize(main)
result = observe(qprog, Pauli.X(0) + Pauli.X(1))
print(result)

### Solution 11 — Higher-order functions

**Exercise A — a user-defined higher-order function**

In [ ]:
@qfunc
def my_apply_to_all(operand: QCallable[QBit], qarr: QArray[QBit]):
    repeat(qarr.len, lambda i: operand(qarr[i]))


@qfunc
def main(qarr: Output[QArray[QBit, 3]]):
    allocate(qarr)
    my_apply_to_all(lambda t: H(t), qarr)


qprog = synthesize(main)
result = sample(qprog)
display(result)

**Exercise B — using the built-in `qpe`**

In [ ]:
@qfunc
def main(theta: Output[QNum[4, UNSIGNED, 4]]):
    state = QArray()
    allocate(2, state)
    X(state[0])
    X(state[1])
    allocate(theta)
    qpe(unitary=lambda: CRZ(pi, state[0], state[1]), phase=theta)


qprog = synthesize(main)
result = sample(qprog)
display(result)

### Solution 12 — Execution parameters and hybrid execution

**Exercise A — sampling a parametric circuit**

In [ ]:
@qfunc
def main(params: CArray[CReal, 2], qarr: Output[QArray[QBit]]):
    allocate(2, qarr)
    RY(params[0], qarr[0])
    RY(params[1], qarr[1])
    CX(qarr[0], qarr[1])


qprog = synthesize(main)
result = sample(qprog, parameters={"params": [0.2, 1.0]})
display(result)

**Exercise B — an expectation value with `observe`**

In [ ]:
import math

hamiltonian = Pauli.X(0) + 0.5 * Pauli.Z(0) * Pauli.Z(1)
energy = observe(qprog, hamiltonian, parameters={"params": [math.pi / 4, math.pi / 3]})
print(f"<H> = {energy}")